In [1]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto_MLDM')
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/SimoRinaldi/crop-spatial-classification.git
    else:
        !cd {REPO_DIR} && git pull
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    !pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato!")
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    print(BASE_DIR)


# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

Mounted at /content/drive
Cloning into 'crop-spatial-classification'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 206 (delta 25), reused 50 (delta 12), pack-reused 142 (from 3)
Receiving objects: 100% (206/206), 184.15 MiB | 36.97 MiB/s, done.
Resolving deltas: 100% (119/119), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 7.8 MB/s eta 0:00:00
Setup ambiente Colab completato!
✅ Collegamento ai dati riuscito! Cartella raw: /content/drive/MyDrive/Progetto_MLDM/data/raw


In [2]:
from pathlib import Path

main_directory = Path(f"{DATA_DIR}/raw/crops_types_yearly_capitanata_03035")

tifs_3035 = {}

for d in main_directory.iterdir():
    if d.is_dir():
        year = d.name
        
        lista_tifs = list(d.rglob("*.tif"))

        tifs_3035[year] = [str(tif) for tif in lista_tifs]

In [4]:
from pathlib import Path
import rioxarray

tifs_4326 = {}

for key, value in tifs_3035.items():
    year_file_list = []
    for v in value:
        file_name = v.replace("03035", "4326").replace("raw", "processed")
        file_path = Path(file_name)
        
        # Se il file riproiettato esiste già, salta la riproiezione
        if file_path.is_file():
            year_file_list.append(file_name)
            continue
        
        # Crea la cartella di destinazione se non esiste
        file_path.parent.mkdir(parents=True, exist_ok=True)
        
        # File GeoTIFF originale (EPSG:3035)
        raster = rioxarray.open_rasterio(v)
        # Riproiezione dell'intero raster in EPSG:4326
        raster_4326 = raster.rio.reproject("EPSG:4326")
        
        raster_4326.rio.to_raster(file_name)
        
        year_file_list.append(file_name)
        
    tifs_4326[key] = year_file_list

In [ ]:
from pathlib import Path
import rasterio
import numpy as np

MAX_SAMPLE_NUMBER = 100

points = []

# Prende la lista di file dell'ultimo gruppo
last_file_list = list(tifs_4326.values())[-1]

for tif in last_file_list:
    with rasterio.open(tif) as dataset:
        full_map = dataset.read(1)

        valid_mask = (full_map > 0) & (full_map < 65534)
        rows, cols = np.where(valid_mask)

        print(full_map.shape)

        indices = np.random.choice(len(rows), size=min(MAX_SAMPLE_NUMBER, len(rows)), replace=False)

        for index in indices:
            lon, lat = dataset.xy(rows[index], cols[index])
            points.append({
                "lat": lat.item(),
                "lon": lon.item(),
                "code": full_map[rows[index], cols[index]].item()
            })

In [ ]:
import json
from pathlib import Path

# Percorso del file JSON nella cartella di lavoro
json_path = Path(f'{DATA_DIR}/interim/points.json')

# Salva i punti (formato: [["nome_file", lon, lat], ...])
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(points, f, indent=4, ensure_ascii=False)

print(f"✅ Salvati {len(points)} punti in {json_path.resolve()}")

In [ ]:
import pandas as pd
df = pd.read_json(f'{DATA_DIR}/interim/points.json')
conteggio = df['code'].value_counts()
print(conteggio)